# 3/3 — Model (bản chạy được trên Linux, 2 file pcap)

Copy của `GNN4ID_Model.ipynb`. Chạy SAU phần B của `1_GNN4ID_pcap.ipynb`
(đã có graph objects trong `Extracted_Flow_Features/processed/`).

In [ ]:
from Utility.Functions import *
from Utility.Model import *
from Utility.Training import *
from Utility.Additional_Features import *
from torch_geometric.loader import DataLoader
from tqdm import tqdm
import glob
import os
import torch

In [ ]:
# [linux] Toàn bộ path của notebook gốc là Windows (F:/CIC_IOT/...) -> đổi sang path máy này.
# 2 file pcap gốc KHÔNG bị đụng tới: mọi thứ sinh ra nằm trong .../Debug and Trace/nb_run/
import os

BASE          = "/home/tutay/Tutay/Tutay_Sec/XG_NID"
PCAP_SRC      = os.path.join(BASE, "data", "Debug and Trace")              # 2 file pcap input
WORK          = os.path.join(PCAP_SRC, "nb_run")                           # thư mục làm việc
Out_Directory = os.path.join(WORK, "Packet_Level_Data")                    # pcap sau khi đổi tên
Out_path      = os.path.join(WORK, "Extracted_Flow_Features") + os.sep     # csv + graph objects
print("WORK     =", WORK)
print("Out_path =", Out_path)

In [ ]:
Dict_x = {'Benign': 0 ,
          'WebBased': 1,
          'Spoofing': 2,
          'Recon' : 3,
          'Mirai' : 4,
          'Dos' : 5,
          'DDos' : 6,
          'BruteForce': 7
         }

dir = Out_path
Files = glob.glob(os.path.join(Out_path, "train", "*.csv"))
print(Files)

In [ ]:
data_Hetero = NIDSDataset(root=dir, label_dict=Dict_x, filename=Files,
                          skip_processing=True, test=False, single_file=True)
print(len(data_Hetero))
data_Hetero[0]

### Khởi tạo model

In [ ]:
## Arguments for running the model
args = {
    'device': torch.device('cuda' if torch.cuda.is_available() else 'cpu'),
    'hidden_size': 64,
    'epochs': 5,          # [linux] tác giả để 30; 2 pcap ít dữ liệu nên để 5 cho nhanh
    'weight_decay': 1e-5,
    'lr': 0.01,
    'attn_size': 32,
    'eps': 1.0,
}
args

In [ ]:
## Initializing a Data Instance for Model Initialization
data_model = data_Hetero[0].to(args['device'])

## Paper-faithful HGNN with GATConv + edge attributes (Section 3.1.4).
model = HeteroGNN(data_model, args, aggr="mean").to(args['device'])
model

### Train

In [ ]:
train_loader = DataLoader(data_Hetero, batch_size=64, shuffle=True)

# lazy layer (-1 channels) cần 1 lần forward để khởi tạo trọng số trước khi train
init_batch = next(iter(DataLoader(data_Hetero, batch_size=2, shuffle=False))).to(args['device'])
with torch.no_grad():
    model(init_batch.x_dict, init_batch.edge_index_dict, init_batch.edge_attr_dict, init_batch)
print("params:", sum(p.numel() for p in model.parameters()))

In [ ]:
# Trains the paper-faithful HGNN (with edge attributes).
train(train_loader, model, args, args["device"])

### Test

In [ ]:
data_Hetero_test = NIDSDataset(root=dir, label_dict=Dict_x, filename=Files,
                               skip_processing=True, test=True, single_file=True)
test_loader = DataLoader(data_Hetero_test, batch_size=1, shuffle=False)
print(len(data_Hetero_test))

In [ ]:
acc, preds, labels = test_cm(test_loader, model, args['device'])
print("accuracy:", acc)

In [ ]:
## Lưu model
torch.save(model, os.path.join(WORK, "xgnid_2pcap.pth"))